# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [ ]:
#!pip install librosa
#!pip install panns-inference

In [ ]:
# !pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [ ]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [ ]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [ ]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
audios = np.array([a["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [ ]:
!pip install -q panns-inference

In [ ]:
from panns_inference import AudioTagging

# load the default model into the gpu.
model = AudioTagging(checkpoint_path=None, device='mps') # change device to cpu if a gpu is not available

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")

CUDA available: True
GPU: Tesla T4


In [ ]:
from panns_inference import AudioTagging

model = AudioTagging(
    checkpoint_path="/root/panns_data/Cnn14_mAP=0.431.pth",
    device="cuda"
)

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth
GPU number: 1


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
# initialize connection to pinecone (get API key at app.pinecone.io)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"

In [ ]:
!pip install -q pinecone-client==3.1.0

In [ ]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [ ]:
!pip uninstall -y pinecone-client
!pip install -q pinecone==9.1.0

Found existing installation: pinecone-client 3.1.0
Uninstalling pinecone-client-3.1.0:
  Successfully uninstalled pinecone-client-3.1.0


In [ ]:
from google.colab import userdata
from pinecone import Pinecone

PINECONE_API_KEY = userdata.get("PINECONE_API_KEY").strip()

pc = Pinecone(api_key=PINECONE_API_KEY)

print("Available indexes:", pc.list_indexes().names())

Available indexes: ['abstractive-question-answering', 'question-answering']


In [ ]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [ ]:
index_name = "audio-search-demo"

In [ ]:
from google.colab import userdata

PINECONE_API_KEY = userdata.get("PINECONE_API_KEY").strip()

print("API key loaded:", bool(PINECONE_API_KEY))
print("API key length:", len(PINECONE_API_KEY))

API key loaded: True
API key length: 75


In [ ]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=2048, total_vector_count=0, metric='cosine', namespaces=0)

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [ ]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/32 [00:00<?, ?it/s]

DescribeIndexStatsResponse(dimension=2048, total_vector_count=2000, metric='cosine', namespaces=1)

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [ ]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

Query Audio: car_horn


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [ ]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [ ]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

QueryResponse(matches=[ScoredVector(id='400', score=1.0000025, values=[]), ScoredVector(id='1667', score=0.842090964, values=[]), ScoredVector(id='1666', score=0.831717372, values=[])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Mon, 02 Nov 2026 16:56:26 GMT', 'content-type': 'application/json', 'content-length': '217', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '32', 'x-pinecone-request-latency-ms': '140', 'x-envoy-upstream-service-time': '82', 'x-pinecone-response-duration-ms': '142', 'grpc-status': '0', 'server': 'envoy'}))

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results.

In [ ]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [ ]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the databse using the id as an index
    query_audio = data[id]["audio"]["array"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=44100))

In [ ]:
find_similar_audios(1642)

Query Audio:


Result:


Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [ ]:
find_similar_audios(452)

Query Audio:


Result:


And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [ ]:
#!wget https://storage.googleapis.com/audioset/miaow_16k.wav

We can load the audio into a Numpy array as follows:

In [ ]:
import os

print("Cartella corrente:", os.getcwd())
print("Contenuto:", os.listdir("."))

if os.path.exists("./data"):
    print("File nella cartella data:", os.listdir("./data"))
else:
    print("La cartella data non esiste.")

Cartella corrente: /content
Contenuto: ['.config', 'sample_data']
La cartella data non esiste.


In [ ]:
import librosa

a, _ = librosa.load("./data/miaow_16k.wav", sr=44100)
Audio(a, rate=44100)

/tmp/ipykernel_10835/3140143619.py:3: UserWarning: PySoundFile failed. Trying audioread instead.
  a, _ = librosa.load("./data/miaow_16k.wav", sr=44100)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: './data/miaow_16k.wav'

Now we generate the embeddings for this audio and query the Pinecone index.

In [ ]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [ ]:
pc.delete_index(index_name)

In [37]:
from datasets import load_dataset

# Load GTZAN music dataset from Parquet
music_dataset = load_dataset(
    "ParitKansal/marsyas-gtzan",
    split="train"
)

print("Number of tracks:", len(music_dataset))
print("Dataset features:", music_dataset.features)

README.md:   0%|          | 0.00/630 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  441MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  429MB            

data/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  436MB            

data/train-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/999 [00:00<?, ? examples/s]

Number of tracks: 999
Dataset features: {'file': Value('string'), 'audio': Audio(sampling_rate=22050, decode=True, num_channels=None, stream_index=None), 'genre': ClassLabel(names=['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])}


In [38]:
from IPython.display import Audio, display

# Select the first track
sample = music_dataset[0]

# Display metadata
print("File:", sample["file"])
print("Genre:", music_dataset.features["genre"].int2str(sample["genre"]))

# Decode audio
audio_obj = sample["audio"]

if hasattr(audio_obj, "get_all_samples"):
    decoded = audio_obj.get_all_samples()
    waveform = decoded.data.squeeze().cpu().numpy()
    sample_rate = decoded.sample_rate
else:
    waveform = audio_obj["array"]
    sample_rate = audio_obj["sampling_rate"]

print("Sampling rate:", sample_rate)
print("Audio shape:", waveform.shape)
print("Duration:", len(waveform) / sample_rate, "seconds")

# Play the track
display(Audio(waveform, rate=sample_rate))

File: C:\Users\xelpmoc\.cache\huggingface\datasets\downloads\extracted\34ed3ff362ee8d2b8e650be17a5643e8a91e2f825b67f3c2dd42604129683e77\genres\blues\blues.00000.wav
Genre: blues
Sampling rate: 22050
Audio shape: (661794,)
Duration: 30.013333333333332 seconds


In [39]:
import numpy as np
import librosa

# Resample audio to 32 kHz
audio_32k = librosa.resample(
    waveform.astype(np.float32),
    orig_sr=sample_rate,
    target_sr=32000
)

print("Original sample rate:", sample_rate)
print("New sample rate: 32000")
print("New audio shape:", audio_32k.shape)

# Generate one test embedding
_, sample_embedding = model.inference(audio_32k[None, :])

print("Embedding shape:", sample_embedding.shape)

Original sample rate: 22050
New sample rate: 32000
New audio shape: (960427,)
Embedding shape: (1, 2048)


In [40]:
import time

music_index_name = "music-recommendations-gtzan"

# Create a new index if it does not already exist
if music_index_name not in pc.list_indexes().names():

    pc.create_index(
        name=music_index_name,
        dimension=2048,
        metric="cosine",
        spec=spec
    )

    # Wait until the index is ready
    while not pc.describe_index(music_index_name).status["ready"]:
        time.sleep(1)

# Connect to the music index
music_index = pc.Index(music_index_name)

# Check index statistics
music_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=2048, total_vector_count=0, metric='cosine', namespaces=0)

In [42]:
import numpy as np
import librosa
import os
from tqdm.auto import tqdm

# Configuration
target_sr = 32000
target_duration = 30
target_length = target_sr * target_duration
batch_size = 8

def prepare_audio(sample):
    audio_obj = sample["audio"]

    # Decode audio
    if isinstance(audio_obj, dict):
        waveform = audio_obj["array"]
        sr = audio_obj["sampling_rate"]
    else:
        decoded = audio_obj.get_all_samples()
        waveform = decoded.data.squeeze().cpu().numpy()
        sr = decoded.sample_rate

    # Convert stereo to mono if necessary
    if waveform.ndim > 1:
        waveform = librosa.to_mono(waveform)

    # Resample to 32 kHz
    waveform = librosa.resample(
        waveform.astype(np.float32),
        orig_sr=sr,
        target_sr=target_sr
    )

    # Make all tracks exactly 30 seconds
    if len(waveform) > target_length:
        waveform = waveform[:target_length]
    elif len(waveform) < target_length:
        waveform = np.pad(
            waveform,
            (0, target_length - len(waveform))
        )

    return waveform.astype(np.float32)


# Process tracks in batches
for start in tqdm(range(0, len(music_dataset), batch_size)):

    end = min(start + batch_size, len(music_dataset))

    batch_samples = [
    music_dataset[idx]
    for idx in range(start, end)
]

    # Prepare audio files
    batch_audio = [
        prepare_audio(sample)
        for sample in batch_samples
    ]

    batch_audio = np.stack(batch_audio)

    # Generate embeddings
    _, embeddings = model.inference(batch_audio)

    # Prepare vectors and metadata
    vectors = []

    for j, embedding in enumerate(embeddings):
        idx = start + j
        sample = batch_samples[j]

        vectors.append({
            "id": str(idx),
            "values": embedding.tolist(),
            "metadata": {
                "genre": music_dataset.features["genre"].int2str(
                    sample["genre"]
                ),
                "filename": str(sample["file"]).replace("\\", "/").split("/")[-1]
            }
        })

    # Upload vectors to Pinecone
    music_index.upsert(vectors=vectors)


# Verify uploaded tracks
music_index.describe_index_stats()

  0%|          | 0/125 [00:00<?, ?it/s]

DescribeIndexStatsResponse(dimension=2048, total_vector_count=999, metric='cosine', namespaces=1)

In [43]:
# Select a reference track
track_id = 0

# Search for the 6 most similar tracks
results = music_index.query(
    id=str(track_id),
    top_k=6,
    include_metadata=True
)

# Display recommendations
for match in results["matches"]:
    print(
        "ID:", match["id"],
        "| Genre:", match["metadata"]["genre"],
        "| Similarity:", round(match["score"], 3),
        "| File:", match["metadata"]["filename"]
    )

ID: 0 | Genre: blues | Similarity: 1.0 | File: blues.00000.wav
ID: 8 | Genre: blues | Similarity: 0.934 | File: blues.00008.wav
ID: 53 | Genre: blues | Similarity: 0.931 | File: blues.00053.wav
ID: 48 | Genre: blues | Similarity: 0.93 | File: blues.00048.wav
ID: 50 | Genre: blues | Similarity: 0.921 | File: blues.00050.wav
ID: 932 | Genre: rock | Similarity: 0.919 | File: rock.00033.wav


In [44]:
# User music preferences
preferred_genres = ["metal", "hiphop", "rock"]

# Extract all genre labels
genre_labels = [
    music_dataset.features["genre"].int2str(g)
    for g in music_dataset["genre"]
]

# Select 6 representative tracks per preferred genre
seed_ids = []

for genre in preferred_genres:
    genre_indices = [
        i for i, label in enumerate(genre_labels)
        if label == genre
    ]

    selected_positions = np.linspace(
        0,
        len(genre_indices) - 1,
        num=min(6, len(genre_indices)),
        dtype=int
    )

    seed_ids.extend([
        genre_indices[pos]
        for pos in selected_positions
    ])

# Prepare selected audio tracks
seed_audio = np.stack([
    prepare_audio(music_dataset[idx])
    for idx in seed_ids
])

# Generate embeddings
_, seed_embeddings = model.inference(seed_audio)

# Create the user's music profile
profile_vector = np.mean(seed_embeddings, axis=0)

# Normalize the profile vector
profile_vector = profile_vector / np.linalg.norm(profile_vector)

print("Preferred genres:", preferred_genres)
print("Tracks used:", len(seed_ids))
print("User profile dimensions:", profile_vector.shape)

Preferred genres: ['metal', 'hiphop', 'rock']
Tracks used: 18
User profile dimensions: (2048,)


In [45]:
# Search Pinecone using the user's music profile
results = music_index.query(
    vector=profile_vector.tolist(),
    top_k=100,
    include_metadata=True
)

# Exclude tracks already used to build the profile
recommendations = [
    match for match in results["matches"]
    if int(match["id"]) not in seed_ids
][:10]

# Display the recommendations
print("YOUR MUSIC RECOMMENDATIONS\n")

for rank, match in enumerate(recommendations, start=1):
    print(
        f"{rank}. {match['metadata']['filename']}"
        f" | Genre: {match['metadata']['genre']}"
        f" | Similarity: {match['score']:.3f}"
    )

YOUR MUSIC RECOMMENDATIONS

1. metal.00085.wav | Genre: metal | Similarity: 0.918
2. metal.00080.wav | Genre: metal | Similarity: 0.916
3. rock.00091.wav | Genre: rock | Similarity: 0.915
4. rock.00036.wav | Genre: rock | Similarity: 0.914
5. metal.00032.wav | Genre: metal | Similarity: 0.909
6. country.00009.wav | Genre: country | Similarity: 0.909
7. hiphop.00023.wav | Genre: hiphop | Similarity: 0.907
8. pop.00009.wav | Genre: pop | Similarity: 0.906
9. metal.00066.wav | Genre: metal | Similarity: 0.906
10. metal.00045.wav | Genre: metal | Similarity: 0.906


In [46]:
from IPython.display import Audio, display

# Listen to the top 5 recommendations
for rank, match in enumerate(recommendations[:5], start=1):

    track_id = int(match["id"])
    sample = music_dataset[track_id]

    # Prepare audio
    audio = prepare_audio(sample)

    print(
        f"Recommendation {rank}: "
        f"{match['metadata']['filename']}"
    )
    print(
        f"Genre: {match['metadata']['genre']} "
        f"| Similarity: {match['score']:.3f}"
    )

    # Play track
    display(Audio(audio, rate=32000))

Recommendation 1: metal.00085.wav
Genre: metal | Similarity: 0.918


Recommendation 2: metal.00080.wav
Genre: metal | Similarity: 0.916


Recommendation 3: rock.00091.wav
Genre: rock | Similarity: 0.915


Recommendation 4: rock.00036.wav
Genre: rock | Similarity: 0.914


Recommendation 5: metal.00032.wav
Genre: metal | Similarity: 0.909
